# Synonym RxNorm Acquisition From Precise Ingredients

## Check Python Version

In [1]:
!python --version

Python 3.11.17


## Setup Config

In [2]:
PROJECT_DIR = "../../"

import pathlib
OUTPUT_DIR = pathlib.Path(PROJECT_DIR)
print(f"Dataset Directory: {OUTPUT_DIR}")

# This file contains list of precise ingredients in each ingredient
FILE_RX_NORM_PRECISE_INGREDIENTS = "rxnorm/rx_norm_precise_ingredients.csv"
RX_NORM_PRECISE_INGREDIENTS_PATH = OUTPUT_DIR.joinpath(FILE_RX_NORM_PRECISE_INGREDIENTS)
print(f"All Drug RxNorm Precise Ingredients Location: {RX_NORM_PRECISE_INGREDIENTS_PATH}")

# Normalize form, each precise ingredient will have 1 row
FILE_RX_NORM_MASTER_PRECISE_INGREDIENTS = "rxnorm/rx_norm_normalize_precise_ingredients.csv"
RX_NORM_MASTER_PRECISE_INGREDIENTS_PATH = OUTPUT_DIR.joinpath(FILE_RX_NORM_MASTER_PRECISE_INGREDIENTS)
print(f"All Drug RxNorm Master Precise Ingredients Location: {RX_NORM_MASTER_PRECISE_INGREDIENTS_PATH}")

FILE_RX_PRECISE_INGREDIENT_SYNONYMS = "rxnorm/rx_norm_precise_ingredients_synonyms.csv"
FILE_RX_PRECISE_INGREDIENT_SYNONYMS_PATH = OUTPUT_DIR.joinpath(FILE_RX_PRECISE_INGREDIENT_SYNONYMS)
print(f"All Drug RxNorm Precise Ingredients Location: {FILE_RX_PRECISE_INGREDIENT_SYNONYMS_PATH}")

Dataset Directory: ../..
All Drug RxNorm Precise Ingredients Location: ../../rxnorm/rx_norm_precise_ingredients.csv
All Drug RxNorm Master Precise Ingredients Location: ../../rxnorm/rx_norm_normalize_precise_ingredients.csv
All Drug RxNorm Precise Ingredients Location: ../../rxnorm/rx_norm_precise_ingredients_synonyms.csv


## Parse Precise Ingredient

Currently we store precise ingredients like this

```csv
rxcui,precise_ingredients
2380376,"[{'rx_cui': '2380375', 'name': '(2-(((2-(2-amino-3-hydroxy-2-(hydroxymethyl)propoxy)-1-hydroxypropyl)hydroxyphosphoryl)oxy)-1-hydroxypropyl)phosphonic acid'}]"
1654972,"[{'rx_cui': '1812466', 'name': '.alpha.-(.alpha.-aminopropyl)benzyl alcohol hydrochloride, erythro-(+/-)-'}]"
1424655,"[{'rx_cui': '1546351', 'name': '1,2-dimyristoyl-sn-glycero-3-(phospho-rac-(1-glycerol)) (sodium salt)'}]"
1599275,"[{'rx_cui': '1599274', 'name': ""1,2-dipalmitoyl-sn-glycero-3-phospho-(1'-rac-glycerol), sodium salt""}]"
1426933,"[{'rx_cui': '1426932', 'name': 'distearoylphosphatidylcholine, DL-'}]"
```

RxCui column refer to rx_cui active ingredients, we need to convert this data to master precise ingredient format like below:
```csv
rxcui,rxcui_ingredient,name
2380375,2380376,"(2-(((2-(2-amino-3-hydroxy-2-(hydroxymethyl)propoxy)-1-hydroxypropyl)hydroxyphosphoryl)oxy)-1-hydroxypropyl)phosphonic acid"
1812466,1654972,".alpha.-(.alpha.-aminopropyl)benzyl alcohol hydrochloride, erythro-(+/-)-"
1546351,1424655,"1,2-dimyristoyl-sn-glycero-3-(phospho-rac-(1-glycerol)) (sodium salt)"
1599274,1599275,"1,2-dipalmitoyl-sn-glycero-3-phospho-(1'-rac-glycerol), sodium salt"
1426932,1426933,"distearoylphosphatidylcholine, DL-"
```

In [3]:
import pandas as pd
import ast

df = pd.read_csv(RX_NORM_PRECISE_INGREDIENTS_PATH)

print(f"Input rows: {len(df):,}")

Input rows: 2,480


In [4]:
def parse_precise_ingredients(value):
    if pd.isna(value):
        return []

    if isinstance(value, list):
        return value

    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return []

df["precise_ingredients"] = df["precise_ingredients"].apply(
    parse_precise_ingredients
)

In [5]:
rows = []

for _, row in df.iterrows():
    rxcui_ingredient = str(row["rxcui"])

    for precise in row["precise_ingredients"]:
        if not isinstance(precise, dict):
            continue

        rxcui = precise.get("rx_cui")
        name = precise.get("name")

        if not rxcui or not name:
            continue

        rows.append({
            "rxcui": str(rxcui),
            "rxcui_ingredient": rxcui_ingredient,
            "name": str(name)
        })

In [6]:
master_df = pd.DataFrame(
    rows,
    columns=[
        "rxcui",
        "rxcui_ingredient",
        "name"
    ]
)

master_df = (
    master_df
    .drop_duplicates()
    .sort_values("rxcui")
    .reset_index(drop=True)
)


In [7]:

print(f"Master precise ingredients: {len(master_df):,}")
print(
    f"Unique precise RxCUI: "
    f"{master_df['rxcui'].nunique():,}"
)
print(
    f"Unique ingredient RxCUI: "
    f"{master_df['rxcui_ingredient'].nunique():,}"
)


master_df.to_csv(RX_NORM_MASTER_PRECISE_INGREDIENTS_PATH, index=False)
print(f"Saved to: {RX_NORM_MASTER_PRECISE_INGREDIENTS_PATH}")

print(master_df.head(10))

Master precise ingredients: 3,659
Unique precise RxCUI: 3,659
Unique ingredient RxCUI: 2,480
Saved to: ../../rxnorm/rx_norm_normalize_precise_ingredients.csv
     rxcui rxcui_ingredient                              name
0  1000104             1712               incobotulinumtoxinA
1  1000112             6691       medroxyprogesterone acetate
2  1000146             4083               estradiol cypionate
3  1005920          1005921                ulipristal acetate
4  1006396          1006397           aminopentamide, sulfate
5  1010241             1091            arginine aminobenzoate
6  1010598             7242  naloxone hydrochloride dihydrate
7    10110            10109              streptomycin sulfate
8  1012533          1012534       melarsomine dihydrochloride
9  1012891          1012892          fingolimod hydrochloride


## Synonym Resolver

In [8]:
import time
import requests

class RxNormSynonymResolver:
    ALLOWED_TYPES = {"SY", "FSY"}
    BASE_URL = "https://rxnav.nlm.nih.gov/REST"

    def _get_rxcui(self, drug_name: str):
        try:
            url = f"{self.BASE_URL}/rxcui.json"
    
            response = requests.get(
                url,
                params={"name": drug_name},
                timeout=10
            )
    
            response.raise_for_status()
            data = response.json()
    
            rxnorm_ids = (
                data.get("idGroup", {})
                .get("rxnormId", [])
            )
    
            if not rxnorm_ids:
                return None
    
            return rxnorm_ids[0]
    
        except Exception as e:
            print(f"[RXCUI ERROR] {drug_name}: {e}")
            return None


    def _get_aliases(self, name: str, rxcui: str):
        try:
            url = f"{self.BASE_URL}/rxcui/{rxcui}/proprietary.json"
    
            response = requests.get(url, timeout=10)
            response.raise_for_status()
    
            data = response.json()
    
            proprietary_info = (
                data.get("proprietaryGroup", {})
                .get("proprietaryInfo", [])
            )
    
            aliases = {}

            for item in proprietary_info:
                if item.get("type") not in self.ALLOWED_TYPES:
                    continue
    
                proprietary_name = item.get("name")
                source = item.get("source")
    
                if not proprietary_name:
                    continue
    
                if name.lower() == proprietary_name.lower():
                    continue
    
                key = proprietary_name.lower()

                if key in aliases:
                    a = aliases[key]
                    a["source"].append(source)

                    aliases[key] = a
                else:
                    aliases[key] = {
                        "name": key,
                        "source": [source]
                    }
    
            return {
                "rx_cui": rxcui,
                "name": name,
                "aliases": list(aliases.values())
            }
    
        except Exception as e:
            print(f"[ALIAS ERROR] {rxcui}: {e}")
    
            return None

    def get_synonyms_by_rx_cui(self, rxcui, drug):
        alias_data = self._get_aliases(drug, rxcui)
        if not alias_data:
            return {
                "status": "unknown_aliases",
                "data": {
                    "rxcui": rxcui,
                    "name": drug
                }
            }
            
        return {
            "status": "ok",
            "data": {
                "rxcui": rxcui,
                "name": drug,
                "aliases": alias_data["aliases"]
            }
        }
        
    def get_synonyms(self, name):
        drug = str(name).strip().lower()
    
        rxcui = self._get_rxcui(drug)
        if not rxcui:
            return {
                "status": "unknown_rxcui",
                "data": {
                    "name": drug
                }
            }
    
        alias_data = self._get_aliases(drug, rxcui)
        if not alias_data:
            return {
                "status": "unknown_aliases",
                "data": {
                    "name": drug
                }
            }
            
        return {
            "status": "ok",
            "data": {
                "name": drug, 
                "rxcui": rxcui,
                "aliases": alias_data["aliases"]
            }
        }

## Resolve Synonyms for RxNorm Precise Ingredients

In [9]:
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm


results = []
unknown_aliases = []

# Maksimal 5 request diproses secara paralel
resolver = RxNormSynonymResolver()

with ThreadPoolExecutor(max_workers=5) as executor:

    futures = {
        executor.submit(
            resolver.get_synonyms_by_rx_cui,
            row["rxcui"],
            row["name"]
        ): row
        for idx, row in master_df.iterrows()
    }

    for future in tqdm(
        as_completed(futures),
        total=len(futures)
    ):
        try:
            result = future.result()

            if result["status"] == "ok":
                results.append(result["data"])

            elif result["status"] == "unknown_aliases":
                unknown_aliases.append(result["data"])

        except Exception as e:
            print(f"[ERROR] {futures[future]}: {e}")

100%|████████████████████████████████████████████████████| 3659/3659 [16:01<00:00,  3.80it/s]


In [10]:
# Sort lagi karena parallel processing
# membuat urutan hasil tidak terjamin
results.sort(key=lambda x: x["rxcui"])
unknown_aliases.sort(key=lambda x: x["rxcui"])

# Export result
df_result = pd.DataFrame(results)
df_result.to_csv(FILE_RX_PRECISE_INGREDIENT_SYNONYMS_PATH, index=False)

print(f"Success         : {len(results)}")
print(f"Unknown aliases : {len(unknown_aliases)}")

Success         : 3659
Unknown aliases : 0


## Save unresolved Precise Ingredients Log

In [11]:
import pathlib
OUTPUT_DIR = pathlib.Path(PROJECT_DIR)
print(f"Dataset Directory: {OUTPUT_DIR}")

FILE_FAILED_RES_PRECISE_INGREDIENTS = "notebooks/reports/rx_norm_failed_precise_ingredients.csv"
FILE_FAILED_RES_PRECISE_INGREDIENTS_PATH = OUTPUT_DIR.joinpath(FILE_FAILED_RES_PRECISE_INGREDIENTS)

Dataset Directory: ../..


In [12]:
import pandas as pd

df_unknown = pd.DataFrame(unknown_aliases)
df_unknown.to_csv(FILE_FAILED_RES_PRECISE_INGREDIENTS_PATH, index=False)
print(f"Unresolved RxNorm Precise Ingredients saved at {FILE_FAILED_RES_PRECISE_INGREDIENTS_PATH}")

Unresolved RxNorm Precise Ingredients saved at ../../notebooks/reports/rx_norm_failed_precise_ingredients.csv
